In [3]:
pip install datasets sentencepiece sacrebleu rouge-score torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=9d05bc62c79694e520fcf8a2556492097e00746957047856182b58dd9b4f7142
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [7]:
from datasets import load_dataset
ds=load_dataset("uqa/UQA")
print(ds)

ex=ds["train"][0]
print(ex.keys())

print(ex["question"])
print(ex["answer"])

n_total=len(ds["train"])
n_ans=sum(len(a)>0 for a in ds["train"]["answer"] if a is not None)
print(f"train rows: {n_total}, answerable: {n_ans}")

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 124745
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 16824
    })
})
dict_keys(['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'])
بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
1990 کی دہائی کے آخر میں
train rows: 124745, answerable: 83018


In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"
SENT_DELIMS = "\u06D4\u061F!" 

def split_sentences(text):
  start = 0
  for i, ch in enumerate(text):
    if ch in SENT_DELIMS:
      yield start, i + 1, text[start : i + 1]
      start = i + 1
  if start < len(text):
    yield start, len(text), text[start:]

def make_pair(example, max_src=60, max_tgt=25):
  answer = example["answer"]
  # Skip unanswerable questions
  if len(answer["text"]) == 0:
    return None
  a_start = example["answer_start"]
  a_text = example["answer"].strip()
  context = example["context"]
  for s, e, sent in split_sentences(context):
    # Check if answer starts inside this sentence
    if s <= a_start < e:
      rel = a_start - s  # Offset inside this specific sentence
      # Integrity check: does the text at this slice match the answer text?
      if sent[rel : rel + len(a_text)] != a_text:
        return None  # Offset mismatch -> skip
      # Insert the <ans> tags
      src = (
          sent[:rel]
          + " "
          + ANS_OPEN
          + " "
          + a_text
          + " "
          + ANS_CLOSE
          + " "
          + sent[rel + len(a_text) :]
      ).strip()
      # Normalise extra whitespaces
      src = " ".join(src.split())
      tgt = " ".join(example["question"].split())
      # Apply length filters (whitespace tokens)
      src_len = len(src.split())
      tgt_len = len(tgt.split())
      if src_len > max_src or tgt_len > max_tgt:
        return None
      return src, tgt
  return None


In [12]:
def build_split(split, out_path):
  pairs = [p for p in map(make_pair, split) if p is not None]
  with open(out_path, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(
        f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\"
    )
    w.writerows(pairs)
  print(f"{out_path}: {len(pairs)} pairs")
  return pairs
# Build train and valid
train_pairs = build_split(ds["train"], "train.tsv")
valid_pairs = build_split(ds["validation"], "valid.tsv")
# Build Wiki-UQA test
wiki_split = wiki_ds["test"] if "test" in wiki_ds else wiki_ds["validation"]
wiki_pairs = build_split(wiki_split, "wiki_test.tsv")

#inspecting the first 3 pairs
for i in range(3):
  src, tgt = train_pairs[i]
  print(f"{i+1} : ")
  print("Source (Sentence + <ans>):", src)
  print("Target (Question):         ", tgt)

TypeError: string indices must be integers, not 'str'